# WM-811K QTran 公平自动调参

本 Notebook 只用训练集和验证集选择配置，**调参期间不构建、不遍历、不输出测试集指标**。四个模型使用相同数量的候选配置和开发种子。

- 阶段1：每个模型12个候选，单开发种子，共48个任务。
- 阶段2：每个模型验证集前3名，3个开发种子，共36个任务。
- 选择分数：`Macro-F1均值 - 0.25 × Macro-F1标准差`。
- 参数量公平限制：与 QTran 相差不超过 ±5%。

## 0. 服务器准备

先在终端进入项目并启动已有环境，再启动 JupyterLab。不要将下面的 Python 单元格内容粘贴到 Bash 终端。

```bash
cd /home/xuxiaoxi/workspace/QCS/QCS
source .venv/bin/activate
python -m jupyter lab --no-browser --ip=127.0.0.1 --port=8888
```

In [ ]:
from pathlib import Path
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
import matplotlib.pyplot as plt

from qcs_wm811k import MODEL_NAMES, environment_report, parameter_audit
from qcs_fair_tuning import (
    N_STAGE1_TRIALS, STAGE1_SEEDS, STAGE2_SEEDS,
    tuning_config, pending_jobs, run_validation_stage,
    validation_summary, select_stage1_top_trials,
    finalize_validation_selection,
)

pd.set_option('display.max_columns', 100)
environment_report()

In [ ]:
PROJECT_DIR = Path.cwd()
CACHE_PATH = PROJECT_DIR / 'data_cache' / 'wm811k_labeled_32.npz'
SEARCH_DIR = PROJECT_DIR / 'artifacts' / 'fair_validation_search_v2_five_projection'
SPLIT_SEED = 20260815

print('Project:', PROJECT_DIR)
print('Cache exists:', CACHE_PATH.exists(), CACHE_PATH)
print('Search output:', SEARCH_DIR)
assert CACHE_PATH.exists(), 'The WM-811K cache is missing.'

## 1. 搜索空间与参数量审计

这一步不训练，只检查全部 QTran 候选是否仍满足 ±5% 参数量公平限制。Trial 0 保留原 QTran，其余候选检验 Pre-LN、注意力温度、残差缩放、线路深度和量子参数学习率倍率。

In [ ]:
audit_rows = []
for trial_id in range(N_STAGE1_TRIALS):
    config = tuning_config('quantum_transformer', trial_id, 'stage1')
    audit = parameter_audit(config, tolerance=0.05)
    qrow = audit[audit['model'] == 'quantum_transformer'].iloc[0]
    audit_rows.append({
        'trial_id': trial_id,
        'qtran_parameters': int(qrow['parameters']),
        'depth': config.quantum_depth,
        'pre_norm': config.quantum_pre_norm,
        'trainable_stabilizers': config.quantum_trainable_stabilizers,
        'temperature_init': config.quantum_attention_temperature,
        'residual_init': config.quantum_residual_scale,
        'quantum_lr_multiplier': config.quantum_lr_multiplier,
    })
display(pd.DataFrame(audit_rows))

## 2. 阶段1：等预算候选筛选

默认每次运行4个尚未完成的任务。每完成一个任务都会立即写入CSV；如果 Kernel 崩溃，重启后重新运行本单元即会从下一个未完成任务继续。确认稳定后可将 `MAX_JOBS=None` 连续运行全部剩余任务。

In [ ]:
stage1_pending = pending_jobs(SEARCH_DIR, 'stage1')
print(f'Stage 1 pending: {len(stage1_pending)}/48')
print('Next jobs:', stage1_pending[:8])

In [ ]:
MAX_JOBS = 4  # 稳定后可改为 None；出现显存或Kernel问题时改为1
stage1_results = run_validation_stage(
    cache_path=CACHE_PATH,
    artifact_dir=SEARCH_DIR,
    stage='stage1',
    split_seed=SPLIT_SEED,
    max_jobs=MAX_JOBS,
)

In [ ]:
result_path = SEARCH_DIR / 'validation_results.csv'
all_validation_results = pd.read_csv(result_path)
stage1_summary = validation_summary(all_validation_results, 'stage1')
display(stage1_summary)

# 只有48/48任务完成时才会生成阶段2候选文件。
stage1_top = select_stage1_top_trials(SEARCH_DIR, top_k=3)
print('Stage 2 candidates:', stage1_top)

## 3. 阶段2：三种子稳定性复核

阶段2只会运行每个模型在阶段1中的前3个配置，使用开发种子 `(101, 202, 303)`。选择时对高标准差进行惩罚，防止只挑选某一次幸运运行。

In [ ]:
stage2_pending = pending_jobs(SEARCH_DIR, 'stage2')
print(f'Stage 2 pending: {len(stage2_pending)}/36')
print('Next jobs:', stage2_pending[:8])

In [ ]:
MAX_JOBS = 3
stage2_results = run_validation_stage(
    cache_path=CACHE_PATH,
    artifact_dir=SEARCH_DIR,
    stage='stage2',
    split_seed=SPLIT_SEED,
    max_jobs=MAX_JOBS,
)

## 4. 冻结四个模型的最终验证配置

只有阶段2的36个任务全部完成后才运行。该单元会生成 `frozen_validation_selection.json`。生成后不要继续修改搜索空间，也不要根据测试集结果更换配置。

In [ ]:
stage2_summary, frozen_trials = finalize_validation_selection(SEARCH_DIR)
display(stage2_summary)
print('Frozen trial IDs:', frozen_trials)
print('Saved:', SEARCH_DIR / 'frozen_validation_selection.json')

In [ ]:
best_rows = (stage2_summary.sort_values('selection_score', ascending=False)
             .groupby('model', as_index=False).first())
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(best_rows['model'], best_rows['val_macro_f1_mean'],
       yerr=best_rows['val_macro_f1_std'], capsize=4)
ax.set_ylabel('Validation Macro-F1')
ax.set_title('Frozen validation-selected configurations')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(SEARCH_DIR / 'frozen_validation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. 何时进入独立确证实验

只有当冻结后的 QTran 验证集 Macro-F1 均值高于所有基线，且领先幅度达到约0.02、标准差降至约0.03时，再创建独立确证实验。如果未达到，先根据验证集训练曲线和消融结果分析原因，不应查看或重复试验测试集。